In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_lv = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lv_2016_2022_bn.csv")

In [3]:
df_lv = df_lv.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_lv

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,EC,DI,SSB,COC,CPI,EUPC
0,1.24,3.200000,5.100000,7.69,31.000000,-0.91,19.17,21.755948,18.476344,3.493186,...,71.7,-0.01,35.050000,-165.000000,71.655374,7.31,94.107515,0.40,57.0,7.912969e+07
1,4.05,3.000000,5.200000,6.87,31.000000,-0.89,19.78,22.712038,18.670316,4.319359,...,72.6,0.00,129.910000,-62.000000,71.655374,7.25,94.112781,0.50,58.0,8.272256e+07
2,1.32,3.500000,5.500000,6.97,31.000000,-0.78,19.35,23.874288,18.535268,5.125604,...,74.7,-0.01,-14.100000,536.000000,73.507226,7.38,94.113949,0.30,58.0,1.773978e+08
3,3.37,3.600000,5.700000,7.29,31.000000,-0.70,19.60,23.338475,19.808901,1.377746,...,75.5,-0.01,17.510000,-152.000000,73.507226,7.49,94.116764,0.48,56.0,3.811824e+08
4,2.81,3.170769,5.380769,7.22,31.000000,-0.70,20.33,22.155426,20.889693,-2.789964,...,75.1,0.00,-38.580000,-649.000000,73.507226,7.24,94.114435,0.69,57.0,3.115265e+08
5,9.78,3.170769,5.380769,6.79,30.000000,-0.84,21.09,25.722828,21.731000,7.848196,...,72.9,-0.03,-35.700000,-69.357143,72.766485,7.31,94.117799,0.72,59.0,3.004081e+08
6,3.20,3.170769,5.380769,6.78,32.333333,-0.27,20.72,24.212312,20.486894,2.083420,...,75.5,0.01,61.486667,-69.357143,72.766485,7.37,94.117799,0.69,59.0,3.078295e+08


In [5]:
discretised_lv = pd.DataFrame(index=df_lv.index)

for col in df_lv.columns:
    no_unique = df_lv[col].nunique()
    
    if no_unique <= 1:
        discretised_lv[col] = 0
    else:
        try:
            discretised_lv[col] = pd.qcut(
                df_lv[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lv[col] = df_lv[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lv)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    0    1    0     2     0    0     0     0     0      1  ...    0    0   
1    2    0    0     0     0    0     1     0     0      1  ...    0    1   
2    0    2    1     1     0    1     0     1     0      2  ...    1    0   
3    1    2    1     2     0    1     0     1     1      0  ...    2    0   
4    0    0    0     1     0    1     1     0     2      0  ...    1    1   
5    2    0    0     0     0    0     2     2     2      2  ...    0    0   
6    1    0    0     0     1    2     2     2     1      0  ...    2    2   

   EPFRG  GGAG  EC  DI  SSB  COC  CPI  EUPC  
0      1     0   0   0    0    0    0     0  
1      2     2   0   0    0    1    1     0  
2      0     2   1   2    0    0    1     0  
3      1     0   1   2    1    0    0     2  
4      0     0   1   0    1    1    0     2  
5      0     1   0   0    2    2    2     1  
6      2     1   0   1    2    1    2     1

In [6]:
sm = from_pandas(discretised_lv)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_lv.html")

Graphs/01_fully_connected_lv.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_lv.html")

Graphs/01_thresholded_lv.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_lv_1.html")

In [10]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

     From     To    Weight
23    PGP    CPI  2.088404
4     QOR   EBSG  2.004362
9     QOA     DI  1.999998
3     QOR    EII  1.996547
8     QOA    QOR  1.983037
18   PDPS    CPI  1.914422
36    EMP   EUPC  1.859521
52    CPI   GGAG  1.834318
30    EMP    FDI  1.803794
14   PDPS    LFG  1.759146
48     DI    CPI  1.589931
54   EUPC    CPI  1.549531
47     DI    FEM  1.513657
53   EUPC   GDPC  1.505143
20    PGP  GDPCG  1.326924
5     QOR   LBGD  1.273081
27   EBSG   LBGD  1.261717
21    PGP    EIS  1.197492
39    LFG   EBSG  1.021044
16   PDPS     DI  0.999994
12   PDPS    EII  0.988011
22    PGP     EC  0.978743
13   PDPS   GDPC  0.972477
26    EII   EIAP  0.954180
2     QOR  GDPCG  0.934651
41     EC   GGFC  0.920454
50    SSB   GCFP  0.900802
38    FEM    EMP  0.882134
28   GDPC    SSB  0.867015
37    LTU  EPFRG  0.859071
34    EMP  EPFRG  0.857909
25  GDPCG    FDI  0.853352
29   LBGD    LTU  0.826759
24   GCFP    COC  0.821965
40    LFG    COC  0.819177
10   EIAP    FEM  0.817885
0